# Binary Classification on the Moons Dataset

Train an MLP to separate two interleaving half-moon shapes using BCE loss.

This application connects across all of Phase 3:
- **Distributions**: the data comes from two interleaving arcs (geometric, not probabilistic)
- **BCE Loss**: Bernoulli NLL → Cross-Entropy — just implemented
- **Decision boundary**: visualising what the model has learned

We'll use the autograd engine and nn modules built entirely from scratch.

In [ ]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parent) if Path.cwd().name == "apps" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.append(project_root)

import matplotlib.pyplot as plt
import numpy as np
from core.autograd import Value
from core.nn import (
    SGD,
    BCELoss,
    DataLoader,
    Linear,
    ReLU,
    Sequential,
    Sigmoid,
)

## 1. Generate the Moons Dataset

Two interleaving half-circles with controllable noise.

We implement `make_moons` from scratch — no sklearn dependency needed.

In [ ]:
np.random.seed(42)

n_samples = 500
noise = 0.20
n_per_class = n_samples // 2

theta = np.linspace(0, np.pi, n_per_class)

x0 = np.cos(theta)
y0 = np.sin(theta)

x1 = 1 - np.cos(theta)
y1 = 0.5 - np.sin(theta)

x0 += np.random.randn(n_per_class) * noise
y0 += np.random.randn(n_per_class) * noise
x1 += np.random.randn(n_per_class) * noise
y1 += np.random.randn(n_per_class) * noise

X = np.vstack(
    [
        np.column_stack([x0, y0]),
        np.column_stack([x1, y1]),
    ]
)
y = np.vstack(
    [
        np.zeros((n_per_class, 1)),
        np.ones((n_per_class, 1)),
    ]
)

indices = np.random.permutation(n_samples)
X = X[indices]
y = y[indices]

# Split 80/20
split = int(0.8 * n_samples)
indices = np.random.permutation(n_samples)
x_train, x_val = X[indices[:split]], X[indices[split:]]
y_train, y_val = y[indices[:split]], y[indices[split:]]

print(f"Train: {x_train.shape}, Val: {x_val.shape}")
print(f"Class balance: {y_train.mean():.2f}")

In [ ]:
# Quick look at the data
plt.figure(figsize=(6, 4))
plt.scatter(
    x_train[y_train.ravel() == 0, 0],
    x_train[y_train.ravel() == 0, 1],
    s=10,
    alpha=0.7,
    label="class 0",
)
plt.scatter(
    x_train[y_train.ravel() == 1, 0],
    x_train[y_train.ravel() == 1, 1],
    s=10,
    alpha=0.7,
    label="class 1",
)
plt.legend()
plt.title("Moons Dataset")
plt.gca().set_aspect("equal")

## 2. Model Definition

Architecture: `Linear(2, 16) -> ReLU() -> Linear(16, 16) -> ReLU() -> Linear(16, 1) -> Sigmoid()`

A 3-layer MLP (two hidden layers) with Kaiming init.
The final Sigmoid squashes the output to (0, 1) — a valid probability.

In [ ]:
model = Sequential(
    [
        Linear(2, 16),
        ReLU(),
        Linear(16, 16),
        ReLU(),
        Linear(16, 1),
        Sigmoid(),
    ]
)

print(model)

## 3. Loss & Optimizer

BCELoss expects probabilities in (0, 1) — which our Sigmoid output provides.

In [ ]:
loss_fn = BCELoss()
optim = SGD(model.parameters(), lr=0.01)

## 4. Training Loop

The training loop follows the same pattern as the MLP regression notebook:
1. Iterate mini-batches
2. Forward → loss
3. zero_grad → backward → step
4. Track train & val loss

**Note**: BCE loss values are typically larger than MSE (because log becomes very negative as probabilities approach 0), so expect the loss curve to start in the 0.5–1.0 range.

In [ ]:
train_loader = DataLoader(x_train, y_train, batch_size=16, shuffle=True)

epochs = 500
train_losses = []
val_losses = []

for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_x, batch_y in train_loader:
        bx, by = Value(batch_x), Value(batch_y)

        pred = model(bx)
        loss = loss_fn(pred, by)

        optim.zero_grad()
        loss.backward()
        optim.step()

        epoch_loss += loss.data

    train_losses.append(epoch_loss / len(train_loader))

    # Validation
    vx, vy = Value(x_val), Value(y_val)
    vpred = model(vx)
    vloss = loss_fn(vpred, vy)
    val_losses.append(vloss.data)

    if (epoch + 1) % 100 == 0:
        print(
            f"Epoch {epoch + 1:4d} | train loss: {train_losses[-1]:.4f} | val loss: {val_losses[-1]:.4f}"
        )

print(f"Final train loss: {train_losses[-1]:.4f}")
print(f"Final val loss:   {val_losses[-1]:.4f}")

## 5. Loss Curves

In [ ]:
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.legend()
plt.title("Training Progress")

## 6. Decision Boundary

The real test of binary classification: visualise the **decision boundary**.

We'll evaluate the model on a dense 2D grid and plot the contour where
the predicted probability equals 0.5.

In [ ]:
model.eval()

# 1. Create a dense meshgrid over the input space
x_min, x_max = -1.5, 2.5
y_min, y_max = -1.0, 2.0
h = 0.02  # grid resolution
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

# 2. Flatten grid points, run through model, reshape back
grid = np.c_[xx.ravel(), yy.ravel()]  # shape (N, 2)
Z = model(Value(grid)).data  # shape (N, 1)
Z = Z.reshape(xx.shape)  # reshape back to grid

# 3. Plot decision regions (contourf) and boundary (contour line)
plt.contourf(xx, yy, Z, levels=[0, 0.5, 1], colors=["#ffcccc", "#cceeff"], alpha=0.8)
plt.contour(xx, yy, Z, levels=[0.5], colors="k", linewidths=1.5)

# 4. Overlay the training data
plt.scatter(
    x_train[y_train.ravel() == 0, 0],
    x_train[y_train.ravel() == 0, 1],
    s=8,
    alpha=0.6,
    label="class 0",
)
plt.scatter(
    x_train[y_train.ravel() == 1, 0],
    x_train[y_train.ravel() == 1, 1],
    s=8,
    alpha=0.6,
    label="class 1",
)
plt.legend()
plt.title("Decision Boundary")
plt.gca().set_aspect("equal")

## 7. Accuracy

Compute classification accuracy on the validation set.
Predictions > 0.5 are class 1, else class 0.

In [ ]:
# Predictions > 0.5 are class 1, else class 0
# vx and y_val are still in scope from the training-loop cell
pred = model(vx)
pred_label = (pred.data > 0.5).astype(float)
accuracy = (pred_label == y_val).mean()
print(f"Validation accuracy: {accuracy * 100:.1f}%")